# PSO vs A* vs Dijkstra — manual comparison (notebook)

Edit parameters in the code cell, run all cells, and inspect printed metrics plus **matplotlib / seaborn** charts.

Implementations: `python_app/algorithms/` (same as the Flask server).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
# Support cwd = python_app or repo root
candidates = [ROOT, ROOT / "python_app"]
for p in candidates:
    if (p / "algorithms").is_dir():
        sys.path.insert(0, str(p.resolve()))
        break

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from algorithms import generate_grid, clear_corridor, run_astar, run_dijkstra, run_pso
from plots import build_comparison_png_bytes, build_analysis_text

sns.set_theme(style="whitegrid", context="notebook")

# -------- Edit these values --------
GRID_SIZE = 18
OBSTACLE_DENSITY = 0.22
RANDOM_SEED = 7

START = (1, 1)
END = (GRID_SIZE - 2, GRID_SIZE - 2)

PSO_PARTICLES = 25
PSO_ITERATIONS = 80
PSO_WAYPOINTS = 5
PSO_SEED = 42
# -----------------------------------

grid = generate_grid(GRID_SIZE, GRID_SIZE, OBSTACLE_DENSITY, seed=RANDOM_SEED)
clear_corridor(grid, START[0], START[1], 1)
clear_corridor(grid, END[0], END[1], 1)

sx, sy = START
ex, ey = END

res_astar = run_astar(grid, sx, sy, ex, ey)
res_dij = run_dijkstra(grid, sx, sy, ex, ey)
res_pso = run_pso(
    grid, sx, sy, ex, ey,
    num_particles=PSO_PARTICLES,
    num_waypoints=PSO_WAYPOINTS,
    max_iterations=PSO_ITERATIONS,
    seed=PSO_SEED,
)

results = [res_astar, res_dij, res_pso]

print(f"{'Algorithm':<12} {'Path?':>6} {'Dist/Len':>12} {'ms':>10} {'Nodes/Eval':>12}")
print("-" * 56)
for r in results:
    d = r.get("distance")
    dstr = f"{d:.3f}" if d is not None and np.isfinite(d) else "—"
    print(
        f"{r['algorithm']:<12} {str(bool(r.get('path'))):>6} {dstr:>12} {r['timeMs']:>10.3f} {r['nodesExplored']:>12}"
    )

print("\n--- Text analysis (same helper as Flask) ---\n")
print(build_analysis_text(results))

# For seaborn bar plots in the next cell
names = [r["algorithm"] for r in results]
dists = [float(r["distance"]) if r.get("distance") is not None and np.isfinite(r["distance"]) else 0.0 for r in results]
times = [r["timeMs"] for r in results]
nodes = [r["nodesExplored"] for r in results]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
pal = ["#38a169", "#3b82f6", "#d4a017"]

sns.barplot(x=names, y=dists, hue=names, palette=pal, ax=axes[0], legend=False)
axes[0].set_title("Path cost / length")

sns.barplot(x=names, y=times, hue=names, palette=pal, ax=axes[1], legend=False)
axes[1].set_title("Time (ms)")

sns.barplot(x=names, y=nodes, hue=names, palette=pal, ax=axes[2], legend=False)
axes[2].set_title("Nodes / evaluations")

plt.tight_layout()
plt.show()

In [ ]:
hist = res_pso.get("fitnessHistory") or []
if hist:
    plt.figure(figsize=(8, 4))
    sns.lineplot(x=np.arange(1, len(hist) + 1), y=hist, marker="o", color="#d4a017")
    plt.xlabel("Iteration")
    plt.ylabel("Best fitness")
    plt.title("PSO: best fitness vs iteration")
    plt.show()
else:
    print("No fitness history")

In [ ]:
from IPython.display import Image, display

png = build_comparison_png_bytes(results)
display(Image(data=png))